In [6]:
import boto3
import lazycogs
from odc.geo.geobox import GeoBox
from affine import Affine
import rustac
from obstore.store import S3Store

geobox = GeoBox(
    shape=(10, 10),
    affine=Affine(5000.0, 0.0, 690400.0, 0.0, -5000.0, 4660400.0),
    crs="EPSG:32614"
)

# 1. Fetch the frozen credentials directly from boto3
session = boto3.Session(profile_name="cdse")
creds = session.get_credentials().get_frozen_credentials()

# # 2. Build the configuration dictionary for obstore
# obstore_config = {
#     "aws_access_key_id": creds.access_key,
#     "aws_secret_access_key": creds.secret_key,
#     # "aws_endpoint_url": "https://eodata.dataspace.copernicus.eu",
#     "aws_virtual_hosted_style_request": "false",
#     "aws_region": session.region_name or "default",
# }
# # If your profile uses temporary STS credentials, pass the token too
# if creds.token:
#     obstore_config["aws_session_token"] = creds.token

obstore_config = {
    "access_key_id": creds.access_key,
    "secret_access_key": creds.secret_key,
    "endpoint": "https://eodata.dataspace.copernicus.eu",  # Required for CDSE
    "virtual_hosted_style_request": "false",
    "region": session.region_name or "default",
}

if creds.token:
    obstore_config["token"] = creds.token



PARQUET = "items.parquet"
items = await rustac.search(
        # PARQUET,
        href="https://stac.dataspace.copernicus.eu/v1/",
        collections=["clms_lst_global_3km_hourly_v3_cog"],
        datetime="2021-03-27",
        bbox=list(geobox.boundingbox.to_crs(4326)),
        limit=1,
    )

store = S3Store(
    bucket="eodata",
    config=obstore_config,
)

In [7]:
from async_geotiff import GeoTIFF
from urllib.parse import urlparse

def strip_bucket(href: str) -> str:
    # href: https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/path/to/file.tif
    # store is rooted at the bucket, so the path is just path/to/file.tif
    return urlparse(href).path.lstrip("/").removeprefix("eodata/")



lazycogs.open_cog(strip_bucket(items[0]["assets"]["lst_lst"]["href"]), store=store)

<xarray.DataArray (band: 1, y: 5601, x: 14400)> Size: 161MB
array([[[-9999, -9999, -9999, ..., -9999, -9999, -9999],
        [-9999, -9999, -9999, ..., -9999, -9999, -9999],
        [-9999, -9999, -9999, ..., -9999, -9999, -9999],
        ...,
        [-9999, -9999, -9999, ..., -9999, -9999, -9999],
        [-9999, -9999, -9999, ..., -9999, -9999, -9999],
        [-9999, -9999, -9999, ..., -9999, -9999, -9999]]],
      shape=(1, 5601, 14400), dtype=int16)
Coordinates:
  * band         (band) int64 8B 1
  * y            (y) float64 45kB 70.0 69.97 69.95 69.92 ... -69.95 -69.98 -70.0
  * x            (x) float64 115kB -180.0 -180.0 -179.9 ... 179.9 180.0 180.0
    spatial_ref  int64 8B 0
Indexes:
  ┌ x        RasterIndex (crs=EPSG:4326)
  └ y
Attributes:
    grid_mapping:  spatial_ref
    _FillValue:    -9999.0
    scale_factor:  0.01
    add_offset:    273.15